# Test Nugder

Objectif :

- tester Nudger sur une première liste de 20 EAN/ISBN ;
- vérifier la couverture de l’API ;
- analyser rapidement la complétude des métadonnées ;
- capturer les erreurs sans bloquer le notebook ;
- exporter les résultats pour analyse future.

## Import

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

from collectionlens.api.nudger_dataset import (
    load_nudger_dataset,
    build_nudger_search_function,
)
from collectionlens.benchmark.api_benchmark import build_source_table

## Liste ISBN test

In [2]:
isbns = [
    "9782351423554",
    "9791042019396",
    "9782820354488",
    "9782290042298",
    "9782845808331",
    "9782413042341",
    "9782384967421",
    "9791043301087",
    "9782374123035",
    "9782374126173",
    "9791039143431",
    "9791026854920",
    "9791039140652",
    "9791026856283",
    "9791039147156",
    "9782203001190",
    "9782858150045",
    "9791038206229",
    "9782822244787",
    "9791038209657",
]

## Exécution du test

In [3]:
nudger_path = PROJECT_ROOT / "data" / "external" / "nudger" / "open4goods-isbn-dataset.csv"

df_nudger = load_nudger_dataset(
    csv_path=nudger_path,
    isbn_column="isbn",
)

nudger_search_func = build_nudger_search_function(df_nudger)

In [4]:
df_nudger_results = build_source_table(
    source_name="nudger",
    search_func=nudger_search_func,
    isbns=isbns,
)

df_nudger_results

,source,isbn_query,found,error,status_code,source_id,isbn,title,subtitle,authors,...,info_link,canonical_volume_link,industry_identifiers,url,format,offers_count,min_price,currency,last_updated,raw_data
0,nudger,9782351423554,True,None,None,9782351423554,9782351423554,vinland saga - tome,NaN,[],...,https://nudger.fr/9782351423554,https://nudger.fr/9782351423554,[],https://nudger.fr/9782351423554,Tankobon,4,3.9,EUR,1779343236174,"{'isbn': '9782351423554', 'title': 'vinland sa..."
1,nudger,9791042019396,True,None,None,9791042019396,9791042019396,Vinland Saga - Tome 29,NaN,[],...,https://nudger.fr/9791042019396,https://nudger.fr/9791042019396,[],https://nudger.fr/9791042019396,None,1,8.95,EUR,1779338869955,"{'isbn': '9791042019396', 'title': 'Vinland Sa..."
2,nudger,9782820354488,True,None,None,9782820354488,9782820354488,Blue Exorcist T32,NaN,[],...,https://nudger.fr/9782820354488,https://nudger.fr/9782820354488,[],https://nudger.fr/9782820354488,None,4,7.25,EUR,1779339488439,"{'isbn': '9782820354488', 'title': 'Blue Exorc..."
3,nudger,9782290042298,True,None,None,9782290042298,9782290042298,fly - tome 1 : le précepteur du héros,NaN,[],...,https://nudger.fr/9782290042298,https://nudger.fr/9782290042298,[],https://nudger.fr/9782290042298,Poche,0,None,None,1711021443483,"{'isbn': '9782290042298', 'title': 'fly - tome..."
4,nudger,9782845808331,True,None,None,9782845808331,9782845808331,Dragon Quest Tome 1,NaN,[],...,https://nudger.fr/9782845808331,https://nudger.fr/9782845808331,[],https://nudger.fr/9782845808331,Tankobon,1,2.61,EUR,1779341024443,"{'isbn': '9782845808331', 'title': 'Dragon Que..."
5,nudger,9782413042341,True,None,None,9782413042341,9782413042341,dragon quest - the adventure of daï t01,NaN,[],...,https://nudger.fr/9782413042341,https://nudger.fr/9782413042341,[],https://nudger.fr/9782413042341,Tankobon,3,2.55,EUR,1779339019935,"{'isbn': '9782413042341', 'title': 'dragon que..."
6,nudger,9782384967421,True,None,None,9782384967421,9782384967421,Les quatre frères Yuzuki Tome 15 (Manga),NaN,[],...,https://nudger.fr/9782384967421,https://nudger.fr/9782384967421,[],https://nudger.fr/9782384967421,None,2,7.2,EUR,1779338019375,"{'isbn': '9782384967421', 'title': 'Les quatre..."
7,nudger,9791043301087,True,None,None,9791043301087,9791043301087,Air Gear Unlimited Tome 07 (Manga),NaN,[],...,https://nudger.fr/9791043301087,https://nudger.fr/9791043301087,[],https://nudger.fr/9791043301087,None,2,10.9,EUR,1779338042634,"{'isbn': '9791043301087', 'title': 'Air Gear U..."
8,nudger,9782374123035,True,None,None,9782374123035,9782374123035,bioman,NaN,[],...,https://nudger.fr/9782374123035,https://nudger.fr/9782374123035,[],https://nudger.fr/9782374123035,None,0,None,None,1711020580414,"{'isbn': '9782374123035', 'title': 'bioman', '..."
9,nudger,9782374126173,False,no_result,None,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Analyse rapide

In [5]:

coverage_rate = df_nudger_results["found"].mean() * 100

print(f"Taux de couverture Nudger api : {coverage_rate:.2f}%")

Taux de couverture Nudger api : 95.00%


In [6]:

df_nudger_results["found"].value_counts(dropna=False)

found
True     19
False     1
Name: count, dtype: int64

## Analyse des erreurs

In [7]:
df_nudger_results.loc[~df_nudger_results["found"], ["isbn_query", "error"]]

,isbn_query,error
9,9782374126173,no_result


## Complétude des métadonnées

In [8]:

metadata_fields = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "language",
    "description",
    "categories",
    "thumbnail",
]

available_fields = [
    field for field in metadata_fields
    if field in df_nudger_results.columns
]

metadata_completeness = (
    df_nudger_results[available_fields]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

metadata_completeness.columns = ["field", "completion_rate"]

metadata_completeness

,field,completion_rate
0,title,95.0
1,authors,95.0
2,publisher,55.0
3,published_date,0.0
4,language,0.0
5,description,0.0
6,categories,95.0
7,thumbnail,0.0


In [9]:
df_nudger_ok = df_nudger_results[df_nudger_results["found"]].copy()
df_nudger_ko = df_nudger_results[~df_nudger_results["found"]].copy()

display(df_nudger_ok)
display(df_nudger_ko)

,source,isbn_query,found,error,status_code,source_id,isbn,title,subtitle,authors,...,info_link,canonical_volume_link,industry_identifiers,url,format,offers_count,min_price,currency,last_updated,raw_data
0,nudger,9782351423554,True,None,None,9782351423554,9782351423554,vinland saga - tome,NaN,[],...,https://nudger.fr/9782351423554,https://nudger.fr/9782351423554,[],https://nudger.fr/9782351423554,Tankobon,4,3.9,EUR,1779343236174,"{'isbn': '9782351423554', 'title': 'vinland sa..."
1,nudger,9791042019396,True,None,None,9791042019396,9791042019396,Vinland Saga - Tome 29,NaN,[],...,https://nudger.fr/9791042019396,https://nudger.fr/9791042019396,[],https://nudger.fr/9791042019396,None,1,8.95,EUR,1779338869955,"{'isbn': '9791042019396', 'title': 'Vinland Sa..."
2,nudger,9782820354488,True,None,None,9782820354488,9782820354488,Blue Exorcist T32,NaN,[],...,https://nudger.fr/9782820354488,https://nudger.fr/9782820354488,[],https://nudger.fr/9782820354488,None,4,7.25,EUR,1779339488439,"{'isbn': '9782820354488', 'title': 'Blue Exorc..."
3,nudger,9782290042298,True,None,None,9782290042298,9782290042298,fly - tome 1 : le précepteur du héros,NaN,[],...,https://nudger.fr/9782290042298,https://nudger.fr/9782290042298,[],https://nudger.fr/9782290042298,Poche,0,None,None,1711021443483,"{'isbn': '9782290042298', 'title': 'fly - tome..."
4,nudger,9782845808331,True,None,None,9782845808331,9782845808331,Dragon Quest Tome 1,NaN,[],...,https://nudger.fr/9782845808331,https://nudger.fr/9782845808331,[],https://nudger.fr/9782845808331,Tankobon,1,2.61,EUR,1779341024443,"{'isbn': '9782845808331', 'title': 'Dragon Que..."
5,nudger,9782413042341,True,None,None,9782413042341,9782413042341,dragon quest - the adventure of daï t01,NaN,[],...,https://nudger.fr/9782413042341,https://nudger.fr/9782413042341,[],https://nudger.fr/9782413042341,Tankobon,3,2.55,EUR,1779339019935,"{'isbn': '9782413042341', 'title': 'dragon que..."
6,nudger,9782384967421,True,None,None,9782384967421,9782384967421,Les quatre frères Yuzuki Tome 15 (Manga),NaN,[],...,https://nudger.fr/9782384967421,https://nudger.fr/9782384967421,[],https://nudger.fr/9782384967421,None,2,7.2,EUR,1779338019375,"{'isbn': '9782384967421', 'title': 'Les quatre..."
7,nudger,9791043301087,True,None,None,9791043301087,9791043301087,Air Gear Unlimited Tome 07 (Manga),NaN,[],...,https://nudger.fr/9791043301087,https://nudger.fr/9791043301087,[],https://nudger.fr/9791043301087,None,2,10.9,EUR,1779338042634,"{'isbn': '9791043301087', 'title': 'Air Gear U..."
8,nudger,9782374123035,True,None,None,9782374123035,9782374123035,bioman,NaN,[],...,https://nudger.fr/9782374123035,https://nudger.fr/9782374123035,[],https://nudger.fr/9782374123035,None,0,None,None,1711020580414,"{'isbn': '9782374123035', 'title': 'bioman', '..."
10,nudger,9791039143431,True,None,None,9791039143431,9791039143431,Star Wars : La citadelle hurlante,NaN,[],...,https://nudger.fr/9791039143431,https://nudger.fr/9791039143431,[],https://nudger.fr/9791039143431,Album,4,9.99,EUR,1779343852642,"{'isbn': '9791039143431', 'title': 'Star Wars ..."


,source,isbn_query,found,error,status_code,source_id,isbn,title,subtitle,authors,...,info_link,canonical_volume_link,industry_identifiers,url,format,offers_count,min_price,currency,last_updated,raw_data
9,nudger,9782374126173,False,no_result,None,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:

output_dir = PROJECT_ROOT / "data" / "processed" / "nudger"
output_dir.mkdir(parents=True, exist_ok=True)

full_output_path = output_dir / "nudger_results_full.csv"
ok_output_path = output_dir / "nudger_results_ok.csv"
ko_output_path = output_dir / "nudger_results_ko.csv"

df_nudger_results.to_csv(full_output_path, index=False)
df_nudger_ok.to_csv(ok_output_path, index=False)
df_nudger_ko.to_csv(ko_output_path, index=False)

# Premiers constats

Les premiers tests réalisés sur le dataset Nudger montrent des résultats très encourageants dans le cadre du POC CollectionLens.

## Couverture des ouvrages

Le taux de couverture observé sur le mini-dataset atteint environ 95 %, ce qui constitue le meilleur résultat obtenu jusqu’à présent parmi les sources testées.

La majorité des ISBN mangas, BD et comics testés ont été retrouvés dans le dataset.

## Complétude des métadonnées

Le dataset Nudger présente une très bonne complétude sur plusieurs champs importants :

- titre : 95 % ;
- auteurs : 95 % ;
- catégories : 95 % ;
- éditeur : 55 %.

Ces résultats montrent un fort potentiel pour :
- la recommandation ;
- la classification ;
- les systèmes de recherche ;
- les futurs usages RAG.

## Limites identifiées

Certaines métadonnées restent cependant absentes ou incomplètes :

- dates de publication ;
- descriptions ;
- langue ;
- thumbnails.

Le dataset semble davantage orienté :
- commerce ;
- catalogage produit ;
- navigation e-commerce.

## Conclusion POC

Nudger apparaît comme une source extrêmement prometteuse pour CollectionLens, notamment pour :
- la couverture ISBN ;
- les catégories ;
- les usages de recommandation.

Les résultats obtenus justifient :
- des benchmarks sur des volumes plus importants ;
- une analyse plus approfondie de la structure du dataset ;
- l’intégration future dans une stratégie multi-sources ;
- l’évaluation de son intérêt comme source principale de couverture ISBN.